# Data Ingest Notebook

We're using the 'us' region to store our BQ tables so that we can make use of the `%%bigquery` built-in magic when querying the tables.

In [1]:
import time
from google.cloud import storage
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError

In [2]:
project_id = "auspicious-lead-382321"
bucket_name = "movie_data_lpez"
folder_name = "initial_load"
dataset_name = "movie_raw"
region = "us"
bq_client = bigquery.Client(project=project_id)

In the next sections, we load the CSV files into BQ tables. Each section loads a different file.

Refer to the [SDK documentation](https://cloud.google.com/python/docs/reference/bigquery/latest/index.html) for details on `LoadJobConfig`.

In [3]:
def create_load_table(file_name, table_name, schema, delimiter=","):

  uri = "gs://{}/{}/{}".format(bucket_name, folder_name, file_name)
  table_id = "{}.{}.{}".format(project_id, dataset_name, table_name)

  table = bigquery.Table(table_id, schema=schema)
  table = bq_client.create_table(table, exists_ok=True)
  print("Created table {}".format(table.table_id))

  # remove the load_time field from the schema before loading the data,
  # the load_time value will be auto-generated
  del schema[-1]

  job_config = bigquery.LoadJobConfig(
        schema=schema,
        skip_leading_rows=1,
        source_format=bigquery.SourceFormat.CSV,
        write_disposition="WRITE_TRUNCATE",
        field_delimiter=delimiter,
        allow_quoted_newlines=True,
        allow_jagged_rows=True
      )

  load_job = bq_client.load_table_from_uri(uri, table_id, job_config=job_config)
  load_job.result()

  destination_table = bq_client.get_table(table_id)
  print("Loaded {} rows.".format(destination_table.num_rows))

# tmdb_actor

In [19]:
file_name = 'tmdb_actor.csv'
table_name = 'tmdb_actor'

schema1 = [
    bigquery.SchemaField("credit_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("person_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("movie_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("character", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("gender", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("credit_order", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP")
]

create_load_table(file_name, table_name, schema1)

Created table tmdb_actor
Loaded 322632 rows.


# tmdb_character

In [17]:
file_name = 'tmdb_character.csv'
table_name = 'tmdb_character'

schema = [
    bigquery.SchemaField("credit_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("person_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("movie_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("character", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("credit_order", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP")
]

create_load_table(file_name, table_name, schema)

Created table tmdb_character
Loaded 317256 rows.


# tmdb_crew

In [18]:
file_name = 'tmdb_crew.csv'
table_name = 'tmdb_crew'

schema = [
    bigquery.SchemaField("credit_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("person_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("movie_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("gender", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("department", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("known_for_department", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("job", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
]

create_load_table(file_name, table_name, schema)

Created table tmdb_crew
Loaded 634740 rows.


# tmdb_person_info

In [ ]:
file_name = 'tmdb_person_info.csv'
table_name = 'tmdb_person_info'

schema = [
    bigquery.SchemaField("person_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("also_known_as", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("biography", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("birthday", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("deathday", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("gender", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("homepage", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("place_of_birth", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("imdb_id", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("known_for_department", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("popularity", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
]

create_load_table(file_name, table_name, schema)

# tmdb_genre

In [ ]:
file_name = 'tmdb_genre.csv'
table_name = 'tmdb_genre'

schema = [
    bigquery.SchemaField("genre_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
]

create_load_table(file_name, table_name, schema)

# tmdb_movie_genre

In [ ]:
file_name = 'tmdb_movie_genre.csv'
table_name = 'tmdb_movie_genre'

schema = [
    bigquery.SchemaField("movie_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("genre_id", "INTEGER", mode = "REQUIRED"),
    bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP")
]

create_load_table(file_name, table_name, schema)

# tmdb_movie

In [6]:
file_name = 'tmdb_movie.csv'
table_name = 'tmdb_movie'

schema = [
    bigquery.SchemaField("record_date", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("movie_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("title", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("original_language", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("overview", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("popularity", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("release_date", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("genre_ids", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("vote_average", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("vote_count", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
]

create_load_table(file_name, table_name, schema)

Created table tmdb_movie
Loaded 29765 rows.


# imdb_movie

In [ ]:
file_name = 'imdb_movie.csv'
table_name = 'imdb_movie'

schema = [
    bigquery.SchemaField("tconst", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("title", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("release_year", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("runtime_minutes", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("genres", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("directors", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("writers", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("average_rating", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("num_ratings", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("actors", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
]

create_load_table(file_name, table_name, schema)

# tmdb_external_id

In [ ]:
file_name = 'tmdb_external_ids.csv'
table_name = 'tmdb_external_ids'

schema = [
    bigquery.SchemaField("tmdb_id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("imdb_id", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("wikidata_id", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("facebook_id", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("instagram_id", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("twitter_id", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP")
]

create_load_table(file_name, table_name, schema)

# github_theater_location

In [ ]:
file_name = 'github_theater_location.csv'
table_name = 'github_theater_location'

schema = [
    bigquery.SchemaField("id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("address", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("city", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("state", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("zip", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("country", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("theaterChainName", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("location", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP")
]

create_load_table(file_name, table_name, schema)

# amc_snack

In [ ]:
file_name = 'amc_snack.csv'
table_name = 'amc_snack'

schema = [
    bigquery.SchemaField("id", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("calories", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("calories_from_fat", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("total_fat", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("saturated_fat", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("trans_fat", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("polyunsat_fat", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("monounsat_fat", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("cholesterol", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("sodium", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("potassium", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("carbohydrates", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("dietary_fiber", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("sugars", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("protein", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
]

create_load_table(file_name, table_name, schema)

# Verify loads

In [ ]:
%load_ext google.cloud.bigquery

sql = "select table_name from {}.INFORMATION_SCHEMA.TABLES order by table_name".format(dataset_name)
query = (sql)

query_job = bq_client.query(
    query,
    location=region,
)

results = query_job.result()

for table in query_job:
    table_name = table.values()[0]
    print("table:", table_name)

In [ ]:
%%bigquery
select * from movie_raw.tmdb_actor limit 5

In [ ]:
%%bigquery
select * from movie_raw.tmdb_person_info limit 5

In [ ]:
%%bigquery
select * from movie_raw.tmdb_character limit 5

In [ ]:
%%bigquery
select * from movie_raw.tmdb_movie_genre limit 5

In [ ]:
%%bigquery
select * from movie_raw.tmdb_genre limit 5

In [ ]:
%%bigquery
select * from movie_raw.tmdb_movie limit 5

In [ ]:
%%bigquery
select * from movie_raw.imdb_movie limit 5

In [ ]:
%%bigquery
select * from movie_raw.tmdb_external_ids limit 5

In [ ]:
%%bigquery
select * from movie_raw.github_theater_location limit 5

In [ ]:
%%bigquery
select * from movie_raw.amc_snack limit 5